# 5 · Detailed Stats  `[EVAL]`

The **full statistical tables** behind the figures in `0`-`1` — kept here so the analysis notebooks stay figure-led. Everything is full-conversation eval, paired by the 96 shared personas. Thin arms (<3 scored iters) are dropped to avoid NaN rows. For a reader who wants exact numbers.

In [7]:
import sys, os; sys.path.insert(0, os.path.abspath("."))
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 185, "display.max_columns", 50)
import exp3
from exp3 import stats, behavior, training, pref, figures, plots

# ── EDA config — edit these flat globals to control the whole notebook ─────────
cfg = exp3.EdaConfig(
    methods=None, ks=None, arm_labels=None,        # arm filter (None = all on disk)
    metrics=None, selection="all", warmth_only=False,  # metric subset / cross-model selection
    focus_arms=None, focus_metric="Q1Q2",          # default arms/metric for overlay & contrast figs
    panel=None, ncols=None, score_ylim=None, share_y=False,  # plot scales (None = inherit)
    export_group="stats",                         # results/<figures|tables>/stats/  ·  figs=PNG, tables=md+xlsx
)
S = exp3.notebook_setup(cfg)
FOCUS = cfg.focus_arms or sorted(S.SCORES.arm.unique())   # arms to show in overlay/contrast figures

arms on disk: [('PTO_LA0', 11), ('PTO_LA5', 6), ('GRPO_LA0', 11), ('GRPO_LA5', 2)]
scores_long: (32851, 19) | arms scored: ['GRPO_LA0', 'GRPO_LA5', 'PTO_LA0', 'PTO_LA5']
metrics: ['Q1Q2', 'WAI-SR', 'CSQ-8', 'MI-SAT', 'MITI', 'PCT', 'MICI', 'Q1', 'Q2'] | selection: all
exports -> c:\Users\baruc\Desktop\Projects\Thesis_PTO_GRPO\Exp3_PTO_GRPO\eda\results  [group: stats]


## 1 · Main results — each arm vs its own base  `[EVAL]`
**Purpose.** Per (arm × rubric): final (and best) iteration vs base — Δ, paired Cohen's *dz* + label, Wilcoxon *p* (Holm), bootstrap 95% CI, trajectory ρ / slope.

In [8]:
MR = stats.filter_thin_arms(stats.main_results_table(S.SCORES, target="final"), S.SCORES)
display(MR)
exp3.save_table(MR, "main_results_final", caption="Final iteration vs base, per arm x rubric, persona-paired (N=96): dz, Wilcoxon p (Holm), bootstrap CI, trajectory rho/slope. Thin arms dropped.")
MRb = stats.filter_thin_arms(stats.main_results_table(S.SCORES, target="best"), S.SCORES)
exp3.save_table(MRb, "main_results_best", caption="As main_results_final but vs each arm's BEST iteration (selection-sensitivity companion).")

  [stats] dropping thin arms (<3 scored iters): ['GRPO_LA5']


,arm,rubric,base,target_iter,target,delta,dz,effect,wilcoxon_p,ci_low,ci_high,traj_rho,traj_slope,p_holm
0,GRPO_LA0,Q1Q2,3.067,10,3.753,0.686,0.721,medium,0.0000,0.501,0.882,0.196,0.0716,0.0000
1,GRPO_LA0,WAI-SR,2.890,10,3.438,0.549,0.837,large,0.0000,0.424,0.680,0.141,0.0395,0.0000
2,GRPO_LA0,CSQ-8,2.362,10,2.773,0.411,0.593,medium,0.0000,0.272,0.555,0.080,0.0241,0.0000
3,GRPO_LA0,MI-SAT,2.710,10,3.319,0.609,0.592,medium,0.0000,0.413,0.816,0.104,0.0396,0.0000
4,GRPO_LA0,MITI,3.224,10,3.922,0.698,0.823,large,0.0000,0.531,0.870,0.258,0.0784,0.0000
5,GRPO_LA0,PCT,0.487,10,0.574,0.087,0.363,small,0.0010,0.040,0.135,0.048,0.0046,0.0010
6,GRPO_LA0,MICI,0.211,10,0.838,0.626,1.717,large,0.0000,0.551,0.699,0.479,0.0437,0.0000
7,GRPO_LA0,Q1,3.021,10,3.606,0.585,0.577,medium,0.0000,0.392,0.800,0.132,0.0561,0.0000
8,GRPO_LA0,Q2,3.113,10,3.899,0.786,0.824,large,0.0000,0.605,0.973,0.260,0.0871,0.0000
9,PTO_LA0,Q1Q2,3.000,10,4.260,1.259,1.429,large,0.0000,1.086,1.434,0.353,0.1203,0.0000


  [stats] dropping thin arms (<3 scored iters): ['GRPO_LA5']


'c:\\Users\\baruc\\Desktop\\Projects\\Thesis_PTO_GRPO\\Exp3_PTO_GRPO\\eda\\results\\tables\\stats'

## 2 · Repeated-measures omnibus (Friedman)  `[EVAL]`
**Purpose.** Is iteration a real within-persona factor? Friedman χ² + Kendall's *W* per arm × rubric.

In [9]:
FR = pd.DataFrame([stats.friedman_trajectory(S.SCORES, a, m)
                   for a in sorted(S.SCORES.arm.unique()) for m in S.METRICS])
FR = stats.filter_thin_arms(FR, S.SCORES)
display(FR.round(4))
exp3.save_table(FR.round(4), "friedman_omnibus", caption="Friedman repeated-measures omnibus across iterations per arm x rubric (Kendall's W). Thin arms dropped.")

  [stats] dropping thin arms (<3 scored iters): ['GRPO_LA5']


,arm,metric,chi2,p,kendall_w,k_iters,n_personas
0,GRPO_LA0,Q1Q2,312.7165,0.0000,0.3257,11,96
1,GRPO_LA0,WAI-SR,166.3047,0.0000,0.1732,11,96
2,GRPO_LA0,CSQ-8,111.1489,0.0000,0.1158,11,96
3,GRPO_LA0,MI-SAT,129.4716,0.0000,0.1349,11,96
4,GRPO_LA0,MITI,303.5148,0.0000,0.3162,11,96
5,GRPO_LA0,PCT,58.1104,0.0000,0.0605,11,96
6,GRPO_LA0,MICI,335.2027,0.0000,0.3492,11,96
7,GRPO_LA0,Q1,265.6537,0.0000,0.2767,11,96
8,GRPO_LA0,Q2,329.8852,0.0000,0.3436,11,96
9,PTO_LA0,Q1Q2,432.4498,0.0000,0.4505,11,96


'c:\\Users\\baruc\\Desktop\\Projects\\Thesis_PTO_GRPO\\Exp3_PTO_GRPO\\eda\\results\\tables\\stats'

## 3 · Per-arm vs-base, every iteration (paired)  `[EVAL]`
**Purpose.** The full iteration-by-iteration Q1+Q2 vs-base table per arm (each arm vs its OWN base).

In [10]:
for arm in sorted(S.SCORES.arm.unique()):
    if arm in stats.thin_arms(S.SCORES): continue
    PV = stats.paired_vs_base(S.SCORES, arm, "Q1Q2")
    if PV.empty: continue
    print(f"=== {arm}: Q1+Q2 vs base (persona-paired) ==="); display(PV[["iteration","n","mean_delta","dz","p_holm","ci_low","ci_high"]].round(4))
    exp3.save_table(PV[["iteration","n","mean_delta","dz","p","p_holm","ci_low","ci_high"]].round(4), f"{arm}_Q1Q2_vs_base_paired", caption=f"{arm} each iteration vs base on Q1+Q2; persona-paired Wilcoxon, dz, Holm p, bootstrap CI.")

=== GRPO_LA0: Q1+Q2 vs base (persona-paired) ===


,iteration,n,mean_delta,dz,p_holm,ci_low,ci_high
0,1,96,0.2022,0.2129,0.0423,0.0202,0.3889
1,2,96,0.2926,0.3341,0.0013,0.1281,0.4740
2,3,96,0.9267,1.1494,0.0000,0.7694,1.0869
3,4,96,0.9376,1.1403,0.0000,0.7731,1.1017
4,5,96,0.9050,1.1238,0.0000,0.7506,1.0659
5,6,96,0.8988,0.9844,0.0000,0.7120,1.0803
6,7,96,1.0072,1.0968,0.0000,0.8363,1.1962
7,8,96,1.0155,1.2196,0.0000,0.8514,1.1852
8,9,96,0.7407,0.7912,0.0000,0.5553,0.9229
9,10,96,0.6858,0.7207,0.0000,0.5015,0.8817


=== PTO_LA0: Q1+Q2 vs base (persona-paired) ===


,iteration,n,mean_delta,dz,p_holm,ci_low,ci_high
0,1,96,0.2632,0.2690,0.0352,0.0814,0.4545
1,2,96,0.4659,0.4587,0.0000,0.2651,0.6732
2,3,96,0.8145,0.8427,0.0000,0.6185,1.0041
3,4,96,1.0072,1.0782,0.0000,0.8208,1.1967
4,5,96,1.0141,1.0532,0.0000,0.8221,1.2057
5,6,96,1.1536,1.2035,0.0000,0.9693,1.3494
6,7,96,1.1287,1.1216,0.0000,0.9314,1.3231
7,8,96,1.2203,1.2504,0.0000,1.0300,1.4240
8,9,96,1.2381,1.3100,0.0000,1.0483,1.4242
9,10,96,1.2594,1.4288,0.0000,1.0859,1.4336


=== PTO_LA5: Q1+Q2 vs base (persona-paired) ===


,iteration,n,mean_delta,dz,p_holm,ci_low,ci_high
0,1,96,0.1775,0.1629,0.1293,-0.0378,0.3901
1,2,96,0.3048,0.2750,0.0369,0.0911,0.5242
2,3,96,0.6710,0.6714,0.0000,0.4645,0.8743
3,4,96,0.8847,0.8811,0.0000,0.6894,1.0835


## 4 · Cross-arm contrasts — PTO vs GRPO (matched K) & K0 vs K5  `[EVAL]`
**Purpose.** The method and look-ahead contrasts as paired tables at matched iterations (iteration-0 base rows dropped). Positive `mean_delta` = first term higher.

In [11]:
for K in sorted(S.SCORES.K.unique()):
    CMP = stats.paired_method_comparison(S.SCORES, "PTO", "GRPO", K=K)
    CMP = CMP[CMP.iteration > 0] if not CMP.empty else CMP
    if CMP.empty: print(f"K={K}: no common PTO/GRPO iters"); continue
    view = CMP[["iteration","metric","n","mean_delta","dz","p_holm"]].round(4)
    print(f"=== PTO_LA{K} - GRPO_LA{K} (paired; + => PTO higher) ==="); display(view)
    exp3.save_table(view, f"PTO_vs_GRPO_LA{K}_paired", caption=f"PTO_LA{K} - GRPO_LA{K} at matched iterations; persona-paired Wilcoxon + dz + Holm.")
for method in ["PTO", "GRPO"]:
    CMP = stats.paired_k_comparison(S.SCORES, method)
    CMP = CMP[CMP.iteration > 0] if not CMP.empty else CMP
    if CMP.empty: print(f"{method}: K0-vs-K5 not comparable yet"); continue
    view = CMP[["iteration","metric","mean_delta","dz","p_holm"]].round(4)
    print(f"=== {method}: LA0 - LA5 (+ => K0 higher) ==="); display(view)
    exp3.save_table(view, f"{method}_K0_vs_K5_paired", caption=f"{method} K0 - K5 at matched iterations; persona-paired Wilcoxon + dz + Holm.")

=== PTO_LA0 - GRPO_LA0 (paired; + => PTO higher) ===


,iteration,metric,n,mean_delta,dz,p_holm
9,1,Q1Q2,96,-0.0055,-0.0060,1.0000
10,1,WAI-SR,96,-0.0807,-0.1129,1.0000
11,1,CSQ-8,96,-0.0755,-0.1153,1.0000
12,1,MI-SAT,96,-0.1372,-0.1481,0.6570
13,1,MITI,96,0.0547,0.0580,1.0000
...,...,...,...,...,...,...
94,10,MITI,96,0.3516,0.4586,0.0001
95,10,PCT,96,0.0560,0.2875,0.0146
96,10,MICI,96,-0.3463,-0.9893,0.0000
97,10,Q1,96,0.5333,0.7076,0.0000


=== PTO_LA5 - GRPO_LA5 (paired; + => PTO higher) ===


,iteration,metric,n,mean_delta,dz,p_holm
9,1,Q1Q2,96,-0.0907,-0.0945,1.0
10,1,WAI-SR,96,-0.0391,-0.0550,1.0
11,1,CSQ-8,96,0.0156,0.0201,1.0
12,1,MI-SAT,96,0.0174,0.0170,1.0
13,1,MITI,96,-0.0885,-0.0941,1.0
14,1,PCT,96,0.0099,0.0395,1.0
15,1,MICI,96,-0.0056,-0.0164,1.0
16,1,Q1,96,-0.0687,-0.0687,1.0
17,1,Q2,96,-0.1127,-0.1145,1.0


=== PTO: LA0 - LA5 (+ => K0 higher) ===


,iteration,metric,mean_delta,dz,p_holm
9,1,Q1Q2,0.0828,0.0931,1.0000
10,1,WAI-SR,-0.0295,-0.0452,1.0000
11,1,CSQ-8,-0.0104,-0.0150,1.0000
12,1,MI-SAT,-0.0747,-0.0789,1.0000
13,1,MITI,0.1276,0.1514,0.9291
14,1,PCT,-0.0274,-0.1142,1.0000
15,1,MICI,-0.0132,-0.0428,1.0000
16,1,Q1,0.0687,0.0760,1.0000
17,1,Q2,0.0968,0.1032,1.0000
18,2,Q1Q2,0.1583,0.1830,0.5569


=== GRPO: LA0 - LA5 (+ => K0 higher) ===


,iteration,metric,mean_delta,dz,p_holm
9,1,Q1Q2,-0.0025,-0.0028,1.0
10,1,WAI-SR,0.0122,0.0178,1.0
11,1,CSQ-8,0.0807,0.1154,1.0
12,1,MI-SAT,0.0799,0.0867,1.0
13,1,MITI,-0.0156,-0.0165,1.0
14,1,PCT,0.0037,0.0179,1.0
15,1,MICI,-0.0004,-0.0014,1.0
16,1,Q1,0.0042,0.0044,1.0
17,1,Q2,-0.0092,-0.0102,1.0


## 5 · Climb rate, rankings, factor structure  `[EVAL]`
**Purpose.** Q1+Q2 OLS slope + Spearman ρ per arm (climb rate vs endpoint); the cross-model leaderboard; and the rubric PCA (PC1 share → are the 6 rubrics ~one factor?).

In [12]:
SL = stats.filter_thin_arms(pd.DataFrame([stats.trajectory_test(S.SCORES, a, "Q1Q2") for a in sorted(S.SCORES.arm.unique())]), S.SCORES)
display(SL[["arm","metric","spearman_rho","ols_slope","peak_iter","final_iter"]].round(4))
exp3.save_table(SL[["arm","metric","spearman_rho","ols_slope","peak_iter","final_iter"]].round(4), "Q1Q2_slope_by_arm", caption="Q1+Q2 per-iteration OLS slope + Spearman rho per arm (climb rate). Thin arms dropped.")
PCA = pd.DataFrame([{"arm": a, "PC1_pct": round(100*stats.rubric_pca(S.SCORES[S.SCORES.arm==a])["explained_variance_ratio"][0], 1)}
                    for a in sorted(S.SCORES.arm.unique()) if stats.rubric_pca(S.SCORES[S.SCORES.arm==a])["explained_variance_ratio"]])
display(PCA)
exp3.save_table(PCA, "rubric_pca_pc1", caption="Variance explained by PC1 of the rubric scores per arm (dominant PC1 => rubrics ~ one latent factor).")

  [stats] dropping thin arms (<3 scored iters): ['GRPO_LA5']


,arm,metric,spearman_rho,ols_slope,peak_iter,final_iter
0,GRPO_LA0,Q1Q2,0.1960,0.0716,8.0,10.0
1,PTO_LA0,Q1Q2,0.3534,0.1203,10.0,10.0
2,PTO_LA5,Q1Q2,0.2720,0.2263,4.0,4.0


,arm,PC1_pct
0,GRPO_LA0,55.0
1,GRPO_LA5,58.0
2,PTO_LA0,55.6
3,PTO_LA5,56.7


'c:\\Users\\baruc\\Desktop\\Projects\\Thesis_PTO_GRPO\\Exp3_PTO_GRPO\\eda\\results\\tables\\stats'